定义因子 → 注册 → 引擎调度 → 校验 → 计算 → 缓存/入库 → 测试

因子：RSI、MACD、DPO
数据源：../../extracted_data/stk_100.csv
============================================================
流程：
  1. 定义因子函数（带 @validate_ohlcv）
  2. @register_factor 注册到 FACTOR_REGISTRY
  3. FactorEngine 从注册表取因子并调度
  4. validator 校验数据
  5. 因子函数计算（pandas/numpy）
  6. 结果写入 cache / database
  7. tests 验证因子正确性

In [1]:
import os
import time
import sqlite3
import functools
from datetime import datetime
from contextlib import contextmanager
from typing import Optional, Dict, List, Callable, Any

import numpy as np
import pandas as pd

In [2]:
# ==================== 1. 全局因子注册表 ====================
# 键：因子名；值：元信息字典
FACTOR_REGISTRY: Dict[str, dict] = {}

In [3]:
# ==================== 2. 装饰器：因子注册 ====================
def register_factor(
    name: str,
    category: str = "unknown",
    description: str = "",
    params: Optional[Dict] = None,
    tags: Optional[List[str]] = None,
    version: str = "1.0.0",
):
    """
    因子注册装饰器工厂。

    用法：
        @register_factor(name="RSI", category="momentum", params={"period": 14})
        def rsi_factor(df, period=14):
            ...
    """
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)

        # 写入全局注册表（注意使用 dict/list 做浅拷贝，避免外部修改污染）
        FACTOR_REGISTRY[name] = {
            "name": name,
            "category": category,
            "description": description,
            "params": dict(params or {}),
            "tags": list(tags or []),
            "version": version,
            "func": func,                       # 保存原始函数
            "registered_at": datetime.now().isoformat(),
        }

        # 给包装函数附加属性，方便反向查询
        wrapper._factor_name = name
        wrapper._factor_meta = FACTOR_REGISTRY[name]
        return wrapper

    return decorator

In [4]:
# ==================== 3. 装饰器：数据校验 ====================
REQUIRED_COLUMNS = ["trade_date", "open", "high", "low", "close", "vol"]


def validate_ohlcv(func: Callable) -> Callable:
    """
    数据校验装饰器。
    确保传入的是含 OHLCV 的 DataFrame，且不为空、不缺列。
    """
    @functools.wraps(func)
    def wrapper(df: pd.DataFrame, *args, **kwargs):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"{func.__name__}: 输入必须是 pandas DataFrame")
        if df.empty:
            raise ValueError(f"{func.__name__}: DataFrame 为空")
        missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
        if missing:
            raise ValueError(f"{func.__name__}: 缺少列 {missing}")
        return func(df, *args, **kwargs)

    return wrapper

In [18]:
# ==================== 4. 因子定义 ====================

# ---------- 4.1 RSI ----------
@register_factor(
    name="RSI",
    category="momentum",
    description="相对强弱指标（Wilder 平滑）",
    params={"period": 14},
    tags=["momentum", "oscillator"],
)
@validate_ohlcv
def rsi_factor(df: pd.DataFrame, period: int = 14) -> pd.Series:
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder 平滑 ≈ alpha = 1/period 的 EMA
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()

    # avg_loss = 0 时 RS 无意义，先替换成 NaN 避免除 0
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - 100 / (1 + rs)

    # 关键：把「avg_loss 确实为 0（且 avg_gain 有值）」的位置改回 100
    # avg_loss 为 NaN（预热期）时条件为 True，保留原 NaN，不受影响
    rsi = rsi.where(avg_loss != 0, 100.0)

    rsi.name = f"RSI_{period}"
    return rsi


In [6]:
# ---------- 4.2 MACD ----------
@register_factor(
    name="MACD",
    category="momentum",
    description="指数平滑异同移动平均",
    params={"fast": 12, "slow": 26, "signal": 9},
    tags=["momentum", "trend"],
)
@validate_ohlcv
def macd_factor(
    df: pd.DataFrame,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9,
) -> pd.DataFrame:
    """
    EMA_fast = EMA(close, fast)
    EMA_slow = EMA(close, slow)
    DIF      = EMA_fast - EMA_slow
    DEA      = EMA(DIF, signal)
    MACD     = (DIF - DEA) * 2
    """
    ema_fast = df["close"].ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_slow = df["close"].ewm(span=slow, adjust=False, min_periods=slow).mean()
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False, min_periods=signal).mean()
    macd = (dif - dea) * 2

    return pd.DataFrame({"DIF": dif, "DEA": dea, "MACD": macd})

In [7]:
# ---------- 4.3 DPO ----------
@register_factor(
    name="DPO",
    category="trend",
    description="区间震荡指标（去趋势价格震荡）",
    params={"period": 20},
    tags=["trend", "cycle"],
)
@validate_ohlcv
def dpo_factor(df: pd.DataFrame, period: int = 20) -> pd.Series:
    """
    DPO = close - SMA(close, period).shift(period//2 + 1)
    去除长期趋势，突出中周期波动
    """
    shift = period // 2 + 1
    sma = df["close"].rolling(window=period, min_periods=period).mean()
    dpo = df["close"] - sma.shift(shift)
    dpo.name = f"DPO_{period}"
    return dpo

In [8]:
# ==================== 5. 缓存 ====================
class SimpleCache:
    """带 TTL 的内存缓存。"""

    def __init__(self, ttl: int = 300):
        self._store: Dict[str, Any] = {}
        self.ttl = ttl

    @staticmethod
    def make_key(*parts) -> str:
        return "::".join(str(p) for p in parts)

    def get(self, key: str):
        item = self._store.get(key)
        if item is None:
            return None
        value, expire_at = item
        if time.time() > expire_at:
            self._store.pop(key, None)
            return None
        return value

    def set(self, key: str, value: Any):
        self._store[key] = (value, time.time() + self.ttl)

    def clear(self):
        self._store.clear()


In [9]:
# ==================== 6. 数据库 ====================
class FactorDatabase:
    """SQLite 因子结果存储，支持上下文管理与参数化查询。"""

    def __init__(self, db_path: str = ":memory:"):
        self.db_path = db_path
        self._init_table()

    @contextmanager
    def connect(self):
        conn = sqlite3.connect(self.db_path)
        try:
            yield conn
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()

    def _init_table(self):
        with self.connect() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS factor_values (
                    ts_code     TEXT,
                    trade_date  TEXT,
                    factor_name TEXT,
                    field       TEXT,
                    value       REAL,
                    PRIMARY KEY (ts_code, trade_date, factor_name, field)
                )
            """)

    def save_series(
        self,
        ts_code: str,
        factor_name: str,
        series: pd.Series,
        field: str = "value",
    ):
        rows = [
            (ts_code, str(idx), factor_name, field, float(val))
            for idx, val in series.dropna().items()
        ]
        if not rows:
            return
        with self.connect() as conn:
            conn.executemany(
                "INSERT OR REPLACE INTO factor_values "
                "(ts_code, trade_date, factor_name, field, value) "
                "VALUES (?, ?, ?, ?, ?)",
                rows,
            )

    def save_frame(self, ts_code: str, factor_name: str, df: pd.DataFrame):
        for col in df.columns:
            self.save_series(ts_code, factor_name, df[col], field=col)

    def query(self, ts_code: str, factor_name: str) -> pd.DataFrame:
        with self.connect() as conn:
            return pd.read_sql_query(
                "SELECT trade_date, field, value FROM factor_values "
                "WHERE ts_code = ? AND factor_name = ? ORDER BY trade_date, field",
                conn,
                params=(ts_code, factor_name),
            )

In [10]:
# ==================== 7. 引擎 ====================
class FactorEngine:
    """因子计算引擎：从注册表取因子并调度。"""

    def __init__(self, data: pd.DataFrame):
        self.data = data
        self.results: Dict[str, Any] = {}
        self.timings: Dict[str, float] = {}

    def run(self, factor_name: str, **kwargs) -> Any:
        meta = FACTOR_REGISTRY.get(factor_name)
        if meta is None:
            raise ValueError(f"因子未注册: {factor_name}")

        func = meta["func"]
        t0 = time.perf_counter()
        result = func(self.data, **kwargs)
        elapsed = time.perf_counter() - t0

        self.results[factor_name] = result
        self.timings[factor_name] = elapsed
        return result

    def run_all(self, category: Optional[str] = None, **common_kwargs) -> Dict[str, Any]:
        for name, meta in FACTOR_REGISTRY.items():
            if category and meta["category"] != category:
                continue
            self.run(name, **common_kwargs)
        return self.results

    def report(self) -> pd.DataFrame:
        rows = []
        for name, elapsed in self.timings.items():
            meta = FACTOR_REGISTRY[name]
            res = self.results[name]
            shape = res.shape if hasattr(res, "shape") else ()
            rows.append({
                "factor": name,
                "category": meta["category"],
                "version": meta["version"],
                "elapsed_ms": round(elapsed * 1000, 3),
                "shape": str(shape),
            })
        return pd.DataFrame(rows)

In [11]:
# ==================== 8. 数据加载 ====================
def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"数据源不存在: {os.path.abspath(path)}\n"
            f"请确认相对路径是否正确（当前工作目录: {os.getcwd()}）"
        )
    df = pd.read_csv(path)
    # 统一日期格式
    df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.strftime("%Y-%m-%d")
    # 按股票、日期排序
    df = df.sort_values(["ts_code", "trade_date"]).reset_index(drop=True)
    return df

In [24]:
def run_tests():
    print("\n" + "=" * 60)
    print("运行测试 (tests/test_factors.py 简化版)")
    print("=" * 60)

    sample = pd.DataFrame({
        "trade_date": pd.date_range("2024-01-01", periods=60).strftime("%Y-%m-%d"),
        "open":   np.arange(60) + 1.0,
        "high":   np.arange(60) + 2.0,
        "low":    np.arange(60) + 0.5,
        "close":  np.arange(60) + 1.5,
        "vol":    np.full(60, 1000.0),
    })

    engine = FactorEngine(sample)

    # --- 测试 1：RSI ---
    rsi = engine.run("RSI", period=14)
    assert len(rsi) == len(sample), "RSI 长度错误"
    assert rsi.iloc[:14].isna().all(), \
        f"RSI 前 14 个应为 NaN，实际非空={rsi.iloc[:14].notna().sum()}"
    assert rsi.iloc[14:].notna().all(), \
        f"RSI 第 15 个起应有值，实际空值={rsi.iloc[14:].isna().sum()}"
    assert np.allclose(rsi.iloc[14:].values, 100.0), \
        f"递增序列 RSI 应恒为 100，实际={rsi.iloc[-1]}"
    print(f"[PASS] RSI  last={rsi.iloc[-1]:.4f}")

    # --- 测试 2：MACD ---
    macd = engine.run("MACD", fast=12, slow=26, signal=9)
    assert list(macd.columns) == ["DIF", "DEA", "MACD"], "MACD 列名错误"
    assert len(macd) == len(sample), "MACD 长度错误"
    assert macd["DIF"].iloc[:25].isna().all(), "DIF 前 25 个应为 NaN"
    assert macd["DIF"].iloc[25:].notna().all(), "DIF 第 26 个起应有值"
    assert macd["DIF"].iloc[-1] > 0, f"上涨序列 DIF 应为正，实际={macd['DIF'].iloc[-1]}"
    print(f"[PASS] MACD DIF={macd['DIF'].iloc[-1]:.4f}")

    # --- 测试 3：DPO ---
    dpo = engine.run("DPO", period=20)
    assert len(dpo) == len(sample), "DPO 长度错误"
    # (20 - 1) + 11 = 30 → 前 30 个 NaN
    assert dpo.iloc[:30].isna().all(), \
        f"DPO 前 30 个应为 NaN，实际非空={dpo.iloc[:30].notna().sum()}"
    assert dpo.iloc[30:].notna().all(), \
        f"DPO 第 31 个起应有值，实际空值={dpo.iloc[30:].isna().sum()}"
    print(f"[PASS] DPO  last={dpo.iloc[-1]:.4f}")

    # --- 测试 4：校验器缺列 ---
    try:
        @validate_ohlcv
        def dummy(df):
            return df
        dummy(pd.DataFrame({"close": [1, 2, 3]}))
        assert False, "应该抛出 ValueError"
    except ValueError:
        print("[PASS] validate_ohlcv 缺列检测")

    # --- 测试 5：校验器类型 ---
    try:
        @validate_ohlcv
        def dummy2(df):
            return df
        dummy2([1, 2, 3])
        assert False, "应该抛出 TypeError"
    except TypeError:
        print("[PASS] validate_ohlcv 类型检测")

    print("全部测试通过 ✅")

In [25]:
run_tests()


运行测试 (tests/test_factors.py 简化版)
[PASS] RSI  last=100.0000
[PASS] MACD DIF=6.8670
[PASS] DPO  last=20.5000
[PASS] validate_ohlcv 缺列检测
[PASS] validate_ohlcv 类型检测
全部测试通过 ✅


In [26]:
# ==================== 10. 主流程 ====================
def main():
    print("=" * 60)
    print("tdx_factor 全流程示例：RSI / MACD / DPO")
    print("=" * 60)

    # ---- 10.1 加载数据 ----
    data_path = "../../extracted_data/stk_100.csv"
    print(f"\n[1] 加载数据: {data_path}")
    df = load_data(data_path)
    print(f"    行数: {len(df)}, 股票数: {df['ts_code'].nunique()}")
    print(f"    列: {list(df.columns)}")

    # ---- 10.2 初始化缓存 / 数据库 ----
    cache = SimpleCache(ttl=300)
    # db = FactorDatabase(":memory:")  # 想持久化可改成 "factors.db"
    # 【暂时关闭数据库】保留实例化代码，需要时取消注释即可
    # db = FactorDatabase(":memory:")

    # ---- 10.3 展示注册表 ----
    print("\n[2] 已注册因子:")
    for name, meta in FACTOR_REGISTRY.items():
        print(f"    - {name:6s} | {meta['category']:10s} | "
              f"params={meta['params']} | v{meta['version']}")

    # ---- 10.4 逐股票计算 ----
    # 演示：只处理第一只股票，避免输出过长
    first_code = df["ts_code"].iloc[0]
    sub = df[df["ts_code"] == first_code].reset_index(drop=True)

    print(f"\n[3] 计算股票 {first_code}，共 {len(sub)} 条")
    engine = FactorEngine(sub)

    for name in FACTOR_REGISTRY:
        cache_key = SimpleCache.make_key(first_code, name, "latest")
        cached = cache.get(cache_key)
        if cached is not None:
            print(f"    {name}: 命中缓存")
            engine.results[name] = cached
            continue

        result = engine.run(name)
        cache.set(cache_key, result)

        # ---- 入库（暂时注释掉，改用 cache 存储）----
        # if isinstance(result, pd.DataFrame):
        #     db.save_frame(first_code, name, result)
        # else:
        #     db.save_series(first_code, name, result)

    # ---- 10.5 引擎报告 ----
    print("\n[4] 引擎执行报告:")
    print(engine.report().to_string(index=False))

    # ---- 10.6 结果预览 ----
    print(f"\n[5] 因子结果预览（{first_code} 最后 5 行）:")
    preview = pd.DataFrame({
        "trade_date": sub["trade_date"],
        "close": sub["close"],
        "RSI": engine.results["RSI"],
        "DIF": engine.results["MACD"]["DIF"],
        "DEA": engine.results["MACD"]["DEA"],
        "MACD": engine.results["MACD"]["MACD"],
        "DPO": engine.results["DPO"],
    })
    print(preview.tail(5).to_string(index=False))

    # ---- 10.7 从 cache 回读（替代原数据库回读）----
    print("\n[6] 从 cache 回读 RSI（前 3 行）:")
    rsi_cache_key = SimpleCache.make_key(first_code, "RSI", "latest")
    rsi_cached = cache.get(rsi_cache_key)
    if rsi_cached is not None:
        rsi_df = rsi_cached.reset_index()
        rsi_df.columns = ["trade_date", "value"]
        print(rsi_df.head(3).to_string(index=False))
    else:
        print("    缓存未命中")

    # ---- 原数据库回读代码（暂时注释，需要时启用）----
    # print("\n[6] 数据库回读 RSI（前 3 行）:")
    # rsi_db = db.query(first_code, "RSI")
    # print(rsi_db.head(3).to_string(index=False))

    # ---- 10.8 缓存命中演示 ----
    print("\n[7] 缓存命中演示:")
    key = SimpleCache.make_key(first_code, "RSI", "latest")
    print(f"    缓存 key = {key}")
    print(f"    缓存存在 = {cache.get(key) is not None}")

    # ---- 10.9 跑测试 ----
    run_tests()

    print("\n全流程完成 ✅")


if __name__ == "__main__":
    main()

tdx_factor 全流程示例：RSI / MACD / DPO

[1] 加载数据: ../../extracted_data/stk_100.csv
    行数: 24678, 股票数: 100
    列: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']

[2] 已注册因子:
    - RSI    | momentum   | params={'period': 14} | v1.0.0
    - MACD   | momentum   | params={'fast': 12, 'slow': 26, 'signal': 9} | v1.0.0
    - DPO    | trend      | params={'period': 20} | v1.0.0

[3] 计算股票 000005.SZ，共 40 条

[4] 引擎执行报告:
factor category version  elapsed_ms   shape
   RSI momentum   1.0.0       0.839   (40,)
  MACD momentum   1.0.0       0.381 (40, 3)
   DPO    trend   1.0.0       0.207   (40,)

[5] 因子结果预览（000005.SZ 最后 5 行）:
trade_date  close       RSI       DIF       DEA     MACD     DPO
2024-02-28   0.92 56.053324 -0.053481 -0.077479 0.047997 -0.0900
2024-02-29   0.97 60.523347 -0.038239 -0.069631 0.062785 -0.0255
2024-03-01   0.92 54.548197 -0.029850 -0.061675 0.063650 -0.0605
2024-03-04   0.87 49.306033 -0.026926 -0.054725 0.055599 -0.092